<a href="https://colab.research.google.com/github/satyapal2240/satya/blob/main/5_marathi_indian_lang_cogtrans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CogTrans for Indian Languages: Sentiment Analysis
## Marathi

> **Paper:** *CogTrans for Indian Languages: Emotion-Aware Sentiment Analysis Across Marathi*  
> Adapted from: Ahmed et al. (2026), *Information Fusion*, 126(B), 103668.

---
### Instructions
1. Place your CSV files (`tweets-train.csv`, `tweets-valid.csv`, `tweets-test.csv`) in the same directory as this notebook
2. Run all cells in order (Runtime → Run All)

## Section 1 — Environment Setup & Library Installation

### ⚠️ IMPORTANT: Steps for Tomorrow
1. **Open your notebook tomorrow.**
2. **Execute Section 1 (Environment Setup) first.**
3. When a pop-up appears asking you to **"Connect to Google Drive"**, accept it immediately so that all your checkpoints and training runs are backed up in real-time.

In [ ]:
# Install required libraries and setup Google Drive directories
!pip install transformers==4.40.0 datasets torch scikit-learn pandas numpy
!pip install indic-nlp-library sentencepiece sacremoses
!pip install matplotlib seaborn tqdm openai

from google.colab import drive
import os

print("\n[INFO] Mounting Google Drive to preserve training progress...")
try:
    drive.mount('/content/drive')
    # Language-specific directory to avoid future ambiguity with other languages
    DRIVE_DIR = '/content/drive/MyDrive/CogTrans_Backup/marathi'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"[SUCCESS] Progress backup directory is active: {DRIVE_DIR}")
except Exception as e:
    DRIVE_DIR = '.'
    print(f"[WARNING] Google Drive could not be mounted. Progress will be saved locally: {e}")

print('All libraries installed successfully.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.0/321.0 kB 16.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.6 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.0 MB/s eta 0:00:


[INFO] Mounting Google Drive to preserve training progress...
[WARNING] Google Drive could not be mounted. Progress will be saved locally: mount failed
All libraries installed successfully.


In [ ]:
import os, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

Device: cuda
GPU: Tesla T4


## Section 2 — Data Loading

### Local Dataset Upload
Run the cell below to upload your dataset files ( `marathi_data.csv`) directly from your computer.

In [ ]:
from google.colab import files
import os

print("Please upload your CSV files now...")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"User uploaded file \"{filename}\" with length {len(uploaded[filename])} bytes")

Please upload your CSV files now...


Saving tweets-test.csv to tweets-test.csv
Saving tweets-train.csv to tweets-train.csv
Saving tweets-valid.csv to tweets-valid.csv
User uploaded file "tweets-test.csv" with length 1201701 bytes
User uploaded file "tweets-train.csv" with length 6438448 bytes
User uploaded file "tweets-valid.csv" with length 786126 bytes


In [ ]:
import os
import pandas as pd

# Updated configuration mapping the uploaded CSV files to Marathi
LANGUAGE_CONFIG = {
    'marathi': {
        'train': 'tweets-train.csv',
        'valid': 'tweets-valid.csv',
        'test': 'tweets-test.csv',
        'backbone': 'google/muril-base-cased',
        'script': 'Marathi'
    }
}

# Highly comprehensive label mapping targeting multiple potential formats
LABEL_MAP = {
    'positive': 0, 'negative': 1, 'neutral': 2,
    'not-offensive': 2, 'not_offensive': 2, 'not offensive': 2, 'not_offensive ': 2,
    'offensive_targeted': 1, 'offensive_untargeted': 1,
    'offensive_targeted_insult_group': 1, 'offensive_targeted_insult_individual': 1,
    'off': 1, 'not': 2,
    '0': 0, '1': 1, '2': 2
}

INV_LABEL = {0: 'Positive', 1: 'Negative', 2: 'Neutral'}

def robust_read_csv(path, lang):
    # Try to identify separators like semicolon, tab, or comma automatically
    try:
        df = pd.read_csv(path, encoding='utf-8', sep=None, engine='python', on_bad_lines='skip')
    except Exception as e:
        print(f"[DEBUG] Error reading {path} with automatic separator: {e}")
        try:
            df = pd.read_csv(path, encoding='utf-8', sep=',', on_bad_lines='skip')
        except Exception as e2:
            print(f"[DEBUG] Fallback to comma failed: {e2}")
            df = pd.read_csv(path, encoding='utf-8', sep='\t', on_bad_lines='skip')

    # Clean column headers
    df.columns = [str(c).strip().lower() for c in df.columns]
    print(f"[DEBUG] File: {path} | Columns found: {list(df.columns)} | Rows: {len(df)}")

    # Map column headers to text and label
    rename_dict = {}
    for col in df.columns:
        if col in ['text', 'sentence', 'text_clean', 'data', 'tweets', 'tweet']:
            rename_dict[col] = 'text'
        elif col in ['label', 'category', 'sentiment', 'class', 'offensive', 'status']:
            rename_dict[col] = 'label'

    df = df.rename(columns=rename_dict)

    # Positional fallback
    if 'text' not in df.columns or 'label' not in df.columns:
        if df.shape[1] >= 2:
            col1 = df.columns[0]
            col2 = df.columns[1]
            mean_len_1 = df[col1].astype(str).str.len().mean()
            mean_len_2 = df[col2].astype(str).str.len().mean()
            if mean_len_1 > mean_len_2:
                df = df.rename(columns={col1: 'text', col2: 'label'})
            else:
                df = df.rename(columns={col2: 'text', col1: 'label'})
        else:
            df['text'] = ""
            df['label'] = ""

    if 'text' not in df.columns:
        df['text'] = ""
    if 'label' not in df.columns:
        df['label'] = ""

    return df[['text', 'label']]

def load_language_data(lang):
    cfg = LANGUAGE_CONFIG[lang]
    frames = []
    for split in ['train', 'valid', 'test']:
        path = cfg.get(split)
        if path:
            if os.path.exists(path):
                print(f"[INFO] Found split '{split}' at {path}. Reading...")
                df_split = robust_read_csv(path, lang)
                frames.append(df_split)
            else:
                print(f"[WARNING] Split '{split}' file {path} not found in directory.")

    if frames:
        df = pd.concat(frames, ignore_index=True)
        df = df.dropna(subset=['text', 'label'])
        df['text'] = df['text'].astype(str).str.strip()
        df['label'] = df['label'].astype(str).str.strip().str.lower()

        df = df[df['label'].isin(LABEL_MAP.keys())]
        df['label'] = df['label'].map(LABEL_MAP).map(INV_LABEL)

        if len(df) > 0:
            if 'emotion' not in df.columns:
                df['emotion'] = 'unknown'
            print(f'[{lang.upper()}] Successfully loaded {len(df)} samples from your files.')
            return df
        else:
            print(f"[WARNING] Zero matching labels after applying LABEL_MAP in {lang}.")

    print(f'[WARNING] Real data not fully found for {lang} – generating synthetic demo data')
    demo = {
        'text': [f'Sample {lang} text {i}' for i in range(60)],
        'label': (['Positive']*20 + ['Negative']*20 + ['Neutral']*20),
        'emotion': (['joy']*20 + ['anger']*20 + ['no_emotion']*20)
    }
    return pd.DataFrame(demo)

datasets = {lang: load_language_data(lang) for lang in LANGUAGE_CONFIG}
print('\nAll available datasets mapped and loaded successfully.')

[INFO] Found split 'train' at tweets-train.csv. Reading...
[DEBUG] File: tweets-train.csv | Columns found: ['tweet', 'label'] | Rows: 12114
[INFO] Found split 'valid' at tweets-valid.csv. Reading...
[DEBUG] File: tweets-valid.csv | Columns found: ['tweet', 'label'] | Rows: 1500
[INFO] Found split 'test' at tweets-test.csv. Reading...
[DEBUG] File: tweets-test.csv | Columns found: ['tweet', 'label'] | Rows: 2250
[MARATHI] Successfully loaded 10576 samples from your files.

All available datasets mapped and loaded successfully.


## Section 3 — Language-Specific Preprocessing

In [ ]:
import re, unicodedata

# Language-specific stop words adapted for Marathi
STOP_WORDS = {
    'marathi': {'आहे', 'आणि', 'पण', 'वर', 'या', 'ते', 'आहेत', 'हा', 'ही', 'होता', 'होती', 'होते'},
}

def preprocess_text(text, lang):
    if not isinstance(text, str): return ''
    # 1. Unicode normalization
    text = unicodedata.normalize('NFC', text)
    # 2. Lowercase
    text = text.lower()
    # 3. Remove URLs
    text = re.sub(r'http\S+|www\S+', ' URL ', text)
    # 4. Normalize user mentions
    text = re.sub(r'@\w+', '@USER', text)
    # 5. Convert emoji/special symbols to text tokens
    text = re.sub(r'[^\w\s\u0900-\u097F\u0C80-\u0CFF\u0B80-\u0BFF\u0980-\u09FF\u0040]', ' EMOJI ', text)
    # 6. Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # 7. Stop word removal
    tokens = text.split()
    sw = STOP_WORDS.get(lang, set())
    tokens = [t for t in tokens if t not in sw]
    return ' '.join(tokens)

def preprocess_dataset(df, lang):
    df = df.copy()
    df['text_clean'] = df['text'].apply(lambda x: preprocess_text(x, lang))
    df['label_id']   = df['label'].astype(str).str.strip().str.lower().map(LABEL_MAP)
    return df

processed = {lang: preprocess_dataset(df, lang) for lang, df in datasets.items()}
print('Preprocessing complete.')
for lang in processed.keys():
    print(f'\n[{lang.upper()}] Preprocessed Head:')
    display(processed[lang][['text', 'text_clean', 'label', 'label_id']].head(3))

Preprocessing complete.

[MARATHI] Preprocessed Head:


,text,text_clean,label,label_id
3,आपला समाज खूप मोठा आहे. त्यात अनेक घटक अंतर्भू...,आपला समाज खूप मोठा EMOJI त्यात अनेक घटक अंतर्भ...,Negative,1
5,जरी मी देशाच्या सेवेत मृत्यू पावली तरी मला त्य...,जरी मी देशाच्या सेवेत मृत्यू पावली तरी मला त्य...,Negative,1
6,"आज, बारामती शहर परिसरात सुरु असणाऱ्या विविध वि...",आज EMOJI बारामती शहर परिसरात सुरु असणाऱ्या विव...,Negative,1


## Section 4 — QAM: Quantity Augmentation Module

In [ ]:
# ── QAM via synonym replacement (rule-based fallback) ───────────────────
# For full LLM-based augmentation, set USE_LLM = True and supply an OpenAI key.
USE_LLM = False   # Set True if you have an OpenAI API key
OPENAI_KEY = ''   # Set your key here: 'sk-...'
import random

def augment_by_deletion(text, drop_prob=0.15):
    """TG-Prompt1 fallback: randomly drop tokens to create a variation."""
    if not isinstance(text, str): return ''
    tokens = text.split()
    if len(tokens) <= 3: return text
    return ' '.join([t for t in tokens if random.random() > drop_prob])

def augment_dataset(df, lang, n_aug=1):
    """Generate n_aug augmented copies per sample."""
    aug_rows = []
    for _, row in df.iterrows():
        for _ in range(n_aug):
            aug_text = augment_by_deletion(row['text_clean'])
            aug_rows.append({
                'text': aug_text, 'text_clean': aug_text,
                'label': row['label'], 'label_id': row['label_id'],
                'emotion': row['emotion'], 'augmented': True
            })
    aug_df = pd.DataFrame(aug_rows)
    df['augmented'] = False
    combined = pd.concat([df, aug_df], ignore_index=True)
    print(f'[{lang.upper()}] Augmented: {len(df)} → {len(combined)} samples')
    return combined

augmented = {}
for lang, df in processed.items():
    # Use more augmentation for low-resource languages
    n = 2 if lang in ['kannada', 'marathi'] else 1
    augmented[lang] = augment_dataset(df, lang, n_aug=n)

print('\nQAM augmentation complete.')

[MARATHI] Augmented: 10576 → 31728 samples

QAM augmentation complete.


## Section 5 — ECA: Emotion Cognitive Analysis (OCC Model)

In [ ]:
import pandas as pd

# ── Expanded OCC emotion rules supporting Marathi and English code-mixed expressions ──────────────────
EMOTION_LEXICON = {
    'marathi': {
        # Native Marathi Script
        'आनंद': ('Desirable', 'PECI'), 'प्रेम': ('Desirable', 'PECI'),
        'चांगले': ('Praiseworthy', 'PECI'), 'अभिनंदन': ('Desirable', 'PECI'),
        'दुःख': ('Undesirable', 'NECI'), 'राग': ('Blameworthy', 'NECI'),
        'भिती': ('Undesirable', 'NECI'), 'वाईट': ('Blameworthy', 'NECI'),
        'मस्त': ('Praiseworthy', 'PECI'), 'सुंदर': ('Praiseworthy', 'PECI'),
        'खोटे': ('Blameworthy', 'NECI'),

        # English & Transliterated/Code-Mixed equivalents
        'happy': ('Desirable', 'PECI'), 'love': ('Desirable', 'PECI'),
        'good': ('Praiseworthy', 'PECI'), 'great': ('Praiseworthy', 'PECI'),
        'nice': ('Praiseworthy', 'PECI'), 'super': ('Praiseworthy', 'PECI'),
        'congrats': ('Desirable', 'PECI'), 'best': ('Praiseworthy', 'PECI'),
        'sad': ('Undesirable', 'NECI'), 'angry': ('Blameworthy', 'NECI'),
        'bad': ('Blameworthy', 'NECI'), 'hate': ('Blameworthy', 'NECI'),
        'worst': ('Blameworthy', 'NECI'), 'fake': ('Blameworthy', 'NECI'),
        'kill': ('Blameworthy', 'NECI'), 'cry': ('Undesirable', 'NECI'),
        'fear': ('Undesirable', 'NECI'), 'hurt': ('Undesirable', 'NECI'),
        'anand': ('Desirable', 'PECI'), 'prem': ('Desirable', 'PECI'),
        'changle': ('Praiseworthy', 'PECI'), 'vait': ('Blameworthy', 'NECI'),
        'rag': ('Blameworthy', 'NECI')
    },
}

def extract_eca_features(text, lang):
    """Extract PECI/NECI features from text using language lexicon."""
    lexicon = EMOTION_LEXICON.get(lang, {})
    tokens  = text.split()
    peci_score, neci_score = 0, 0
    for token in tokens:
        # Clean token from any trailing punctuation
        clean_token = token.strip('.,!?-~@')
        if clean_token in lexicon:
            _, polarity = lexicon[clean_token]
            if polarity == 'PECI': peci_score += 1
            else:                   neci_score += 1
    total = peci_score + neci_score
    emotion_score = (peci_score - neci_score) / total if total > 0 else 0.0
    dominant = 'PECI' if emotion_score > 0 else ('NECI' if emotion_score < 0 else 'NEUTRAL')
    return {'peci': peci_score, 'neci': neci_score, 'emotion_score': round(emotion_score, 3), 'dominant': dominant}

# Apply ECA to augmented datasets
for lang, df in augmented.items():
    eca_results  = df['text_clean'].apply(lambda t: extract_eca_features(t, lang))
    df['peci']          = eca_results.apply(lambda x: x['peci'])
    df['neci']          = eca_results.apply(lambda x: x['neci'])
    df['emotion_score'] = eca_results.apply(lambda x: x['emotion_score'])
    df['eca_dominant']  = eca_results.apply(lambda x: x['dominant'])

print('ECA features extracted with expanded lexicon.')
for lang in augmented.keys():
    print(f'\nSample ({lang.upper()}):')
    non_zero_samples = augmented[lang][augmented[lang]['emotion_score'] != 0.0]
    if len(non_zero_samples) > 0:
        display(non_zero_samples[['text_clean', 'peci', 'neci', 'emotion_score', 'eca_dominant']].head(5))
    else:
        display(augmented[lang][['text_clean', 'peci', 'neci', 'emotion_score', 'eca_dominant']].head(5))

ECA features extracted with expanded lexicon.

Sample (MARATHI):


,text_clean,peci,neci,emotion_score,eca_dominant
38,वैद्यकीय अधिकारी EMOJI कर्मचारी EMOJI पोलीस EM...,1,0,1.0,PECI
86,राष्ट्रीय चित्रपट पुरस्कार वितरण सोहोळ्यात अने...,1,0,1.0,PECI
96,आदरणीय पवार साहेब मा EMOJI सौ EMOJI प्रतिभाताई...,1,0,1.0,PECI
103,महाराष्ट्राचे मुख्यमंत्री म्हणून शपथ घेतल्याबद...,1,0,1.0,PECI
134,राज्य सांस्कृतिक पुरस्कार संगीत नाटक अकादमी पु...,1,0,1.0,PECI


## Section 6 — T-SR: MuRIL/XLM-R Fine-tuning (CogTrans Backbone)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import pickle
import os

MAX_LEN   = 128
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-5

# Fallback to local if DRIVE_DIR was not set during mounting step
if 'DRIVE_DIR' not in globals():
    DRIVE_DIR = '.'

class IndianLangDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

def train_one_language(lang, df, n_epochs=EPOCHS):
    backbone = LANGUAGE_CONFIG[lang]['backbone']

    # Dynamic language subdirectory configuration
    lang_dir = os.path.join(DRIVE_DIR, lang) if not DRIVE_DIR.endswith(lang) else DRIVE_DIR
    os.makedirs(lang_dir, exist_ok=True)

    model_save_path = os.path.join(lang_dir, f'best_model_{lang}.pt')
    results_save_path = os.path.join(lang_dir, f'results_metrics_{lang}.pkl')

    # Check if we already trained this model and saved results in Google Drive
    if os.path.exists(model_save_path) and os.path.exists(results_save_path):
        print(f'\n[INFO] Found existing backup for [{lang.upper()}] in Google Drive. Loading saved progress...')
        try:
            with open(results_save_path, 'rb') as f:
                saved_res = pickle.load(f)
            # Reconstruct model mapping object
            tokenizer = AutoTokenizer.from_pretrained(backbone)
            model = AutoModelForSequenceClassification.from_pretrained(backbone, num_labels=3)
            model.load_state_dict(torch.load(model_save_path, map_location=DEVICE))
            model.to(DEVICE)
            saved_res['model'] = model
            saved_res['tokenizer'] = tokenizer
            print(f'  ✓ [{lang.upper()}] LOADED  Accuracy={saved_res["test_acc"]:.4f}  Macro-F1={saved_res["test_f1"]:.4f}')
            return saved_res
        except Exception as e:
            print(f'[WARNING] Failed to load saved progress for {lang}: {e}. Retraining...')

    print(f'\n══ Training [{lang.upper()}] with {backbone} ══')
    tokenizer = AutoTokenizer.from_pretrained(backbone)

    # Split
    X = df['text_clean'].tolist()
    y = df['label_id'].tolist()
    X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
    X_val,   X_test, y_val,  y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

    train_ds = IndianLangDataset(X_train, y_train, tokenizer)
    val_ds   = IndianLangDataset(X_val,   y_val,   tokenizer)
    test_ds  = IndianLangDataset(X_test,  y_test,  tokenizer)

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

    model = AutoModelForSequenceClassification.from_pretrained(backbone, num_labels=3)
    model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=n_epochs)

    best_val_f1 = 0.0
    history = {'train_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(n_epochs):
        # ── Train ──
        model.train()
        total_loss = 0
        for batch in tqdm(train_dl, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False):
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            labels    = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            out  = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            loss = out.loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_dl)

        # ── Validate ──
        model.eval()
        preds_val, true_val = [], []
        with torch.no_grad():
            for batch in val_dl:
                out = model(
                    input_ids=batch['input_ids'].to(DEVICE),
                    attention_mask=batch['attention_mask'].to(DEVICE)
                )
                preds_val += out.logits.argmax(-1).cpu().tolist()
                true_val  += batch['label'].tolist()

        val_acc = accuracy_score(true_val, preds_val)
        val_f1  = f1_score(true_val, preds_val, average='macro', zero_division=0)
        history['train_loss'].append(avg_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        scheduler.step()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), model_save_path)
            # Copy locally as fallback
            torch.save(model.state_dict(), f'best_model_{lang}.pt')

        print(f'  Epoch {epoch+1}: loss={avg_loss:.4f}  val_acc={val_acc:.4f}  val_f1={val_f1:.4f}')

    # ── Test ──
    model.load_state_dict(torch.load(model_save_path, map_location=DEVICE))
    model.eval()
    preds_test, true_test = [], []
    with torch.no_grad():
        for batch in test_dl:
            out = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE)
            )
            preds_test += out.logits.argmax(-1).cpu().tolist()
            true_test  += batch['label'].tolist()

    test_acc = accuracy_score(true_test, preds_test)
    test_f1  = f1_score(true_test, preds_test, average='macro', zero_division=0)
    print(f'\n  ✓ [{lang.upper()}] TEST  Accuracy={test_acc:.4f}  Macro-F1={test_f1:.4f}')

    output_res = {
        'lang': lang, 'test_acc': test_acc, 'test_f1': test_f1,
        'history': history,
        'preds': preds_test, 'true': true_test
    }

    # Backup metrics dict to Drive (excluding active model instance to avoid size bloating/pickle issues)
    try:
        with open(results_save_path, 'wb') as f:
            pickle.dump(output_res, f)
        print(f"[INFO] Saved results and weights to Drive: {lang_dir}")
    except Exception as e:
        print(f"[WARNING] Failed to write progress file to Drive: {e}")

    # Append standard model mapping variables for runtime utilization
    output_res['model'] = model
    output_res['tokenizer'] = tokenizer

    return output_res

print('Progress-aware training functions ready.')

Progress-aware training functions ready.


### Section 6.1 — Copying Trained Progress to Google Drive
Run the cell below to copy your local saved weights and metrics directly to your Google Drive backup directory.

In [ ]:
import os
import shutil

# Source paths (local temporary files)
local_model_path = '/content/best_model_marathi.pt'
local_results_dir = '/content/marathi'

# Destination directory in your Google Drive
drive_backup_dir = '/content/drive/MyDrive/CogTrans_Backup/marathi'

print("=== Starting Backup to Google Drive ===")
if os.path.exists('/content/drive'):
    os.makedirs(drive_backup_dir, exist_ok=True)

    # 1. Copy the main best_model_marathi.pt file
    if os.path.exists(local_model_path):
        shutil.copy(local_model_path, os.path.join(drive_backup_dir, 'best_model_marathi.pt'))
        print("✓ Successfully backed up 'best_model_marathi.pt' to your Google Drive!")
    else:
        print("[!] Local 'best_model_marathi.pt' not found in /content. Checking inside subdirectories...")
        fallback_model = '/content/marathi/best_model_marathi.pt'
        if os.path.exists(fallback_model):
            shutil.copy(fallback_model, os.path.join(drive_backup_dir, 'best_model_marathi.pt'))
            print("✓ Successfully backed up 'best_model_marathi.pt' (from subfolder) to your Google Drive!")
        else:
            print("[X] Error: No model weights found locally to copy. Make sure Section 6 has finished training.")

    # 2. Copy the results_metrics_marathi.pkl file
    local_pkl_path = '/content/marathi/results_metrics_marathi.pkl'
    if os.path.exists(local_pkl_path):
        shutil.copy(local_pkl_path, os.path.join(drive_backup_dir, 'results_metrics_marathi.pkl'))
        print("✓ Successfully backed up 'results_metrics_marathi.pkl' to your Google Drive!")
    else:
        print("[!] Evaluation metrics file (.pkl) not found. Run Section 6 first.")

    # 3. Final verification of files in Google Drive
    print("\n=== Verification: Files currently in Google Drive ===")
    print(os.listdir(drive_backup_dir))
else:
    print("[ERROR] Google Drive is not mounted. Please execute Section 1 at the top of the page first.")

=== Starting Backup to Google Drive ===
[!] Local 'best_model_marathi.pt' not found in /content. Checking inside subdirectories...
[X] Error: No model weights found locally to copy. Make sure Section 6 has finished training.
[!] Evaluation metrics file (.pkl) not found. Run Section 6 first.

=== Verification: Files currently in Google Drive ===
['best_model_marathi.pt', 'results_metrics_marathi.pkl']


In [ ]:
import os

drive_backup_dir = '/content/drive/MyDrive/CogTrans_Backup/marathi'

print("=== Google Drive Content Check ===")
if os.path.exists(drive_backup_dir):
    files = os.listdir(drive_backup_dir)
    print(f"Success! Your Google Drive backup folder contains {len(files)} files:\n")
    for idx, f in enumerate(sorted(files), 1):
        size_mb = os.path.getsize(os.path.join(drive_backup_dir, f)) / (1024 * 1024)
        print(f"  {idx}. {f:<30} | Size: {size_mb:.2f} MB")
else:
    print("[ERROR] Backup directory not found on your Google Drive.")

=== Google Drive Content Check ===
Success! Your Google Drive backup folder contains 2 files:

  1. best_model_marathi.pt          | Size: 906.29 MB
  2. results_metrics_marathi.pkl    | Size: 0.02 MB


In [ ]:
# ── Run training for all 5 languages with robust NaN filtering ────────────────
results = {}
for lang in LANGUAGE_CONFIG:
    # Filter out any NaN values from the inputs to prevent train_test_split from crashing
    df_clean = augmented[lang].dropna(subset=['text_clean', 'label_id'])
    # Ensure label_id contains only valid integers
    df_clean['label_id'] = df_clean['label_id'].astype(int)

    if len(df_clean) > 0:
        results[lang] = train_one_language(lang, df_clean)
    else:
        print(f"Skipping training for {lang} due to lack of valid cleaned data.")

print('\nAll languages trained successfully!')


══ Training [MARATHI] with google/muril-base-cased ══


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

...

  Epoch 1: loss=0.4967  val_acc=0.8897  val_f1=0.8893


  Epoch 2: loss=0.2263  val_acc=0.9317  val_f1=0.9317


  Epoch 3: loss=0.1450  val_acc=0.9519  val_f1=0.9519


  Epoch 4: loss=0.0815  val_acc=0.9584  val_f1=0.9584


  Epoch 5: loss=0.0468  val_acc=0.9651  val_f1=0.9651

  ✓ [MARATHI] TEST  Accuracy=0.9664  Macro-F1=0.9664
[INFO] Saved results and weights to Drive: ./marathi

All languages trained successfully!


In [ ]:
from google.colab import drive
import os

# 1. Ensure Google Drive is mounted
try:
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive is mounted.")
except Exception as e:
    print(f"[ERROR] Failed to mount Drive: {e}")

# 2. Correct the DRIVE_DIR to point to your Drive folder
DRIVE_DIR = '/content/drive/MyDrive/CogTrans_Backup/marathi'
globals()['DRIVE_DIR'] = DRIVE_DIR
print(f"[INFO] Backups path successfully pointed to: {DRIVE_DIR}")

# 3. Verify files are indeed there before running again
if os.path.exists(DRIVE_DIR):
    print(f"[SUCCESS] Backup folder verified. Found files: {os.listdir(DRIVE_DIR)}")
else:
    print("[WARNING] Backup directory still not found on Drive. Please verify the path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[SUCCESS] Google Drive is mounted.
[INFO] Backups path successfully pointed to: /content/drive/MyDrive/CogTrans_Backup/marathi
[SUCCESS] Backup folder verified. Found files: ['best_model_marathi.pt', 'results_metrics_marathi.pkl']


In [ ]:
# 4. Re-run training (this will now load from Drive and skip the epochs!)
results = {}
for lang in LANGUAGE_CONFIG:
    df_clean = augmented[lang].dropna(subset=['text_clean', 'label_id'])
    df_clean['label_id'] = df_clean['label_id'].astype(int)

    if len(df_clean) > 0:
        results[lang] = train_one_language(lang, df_clean)
    else:
        print(f"Skipping training for {lang} due to lack of valid cleaned data.")

print('\nDone!')


[INFO] Found existing backup for [MARATHI] in Google Drive. Loading saved progress...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

  ✓ [MARATHI] LOADED  Accuracy=0.9664  Macro-F1=0.9664

Done!


In [ ]:
# Cell content removed to clean up duplicate checks. File checking and backups are handled dynamically in the next cell.

In [ ]:
import os
import shutil

print("=== Step 1: Checking Google Drive Setup ===")
# Safely retrieve DRIVE_DIR from globals or fallback to default
global_drive_dir = globals().get('DRIVE_DIR', '/content/drive/MyDrive/CogTrans_Backup/marathi')

if os.path.exists('/content/drive'):
    os.makedirs(global_drive_dir, exist_ok=True)
    print(f"\n[SUCCESS] Connected to Google Drive!")

    print("\n=== Step 2: Copying saved files from Colab to Google Drive ===")
    local_files = [f for f in os.listdir('.') if f.endswith('.pt') or f.endswith('.pkl') or f.endswith('.zip') or f.endswith('.png')]

    if not local_files:
        print("No local training files (.pt, .pkl, .zip) found in local workspace to back up.")
    else:
        for file_name in local_files:
            dest_path = os.path.join(global_drive_dir, file_name)
            shutil.copy(file_name, dest_path)
            print(f"✓ Copied '{file_name}' to Google Drive ({dest_path})")

    print("\n=== Step 3: Verifying files in your Drive folder ===")
    print(os.listdir(global_drive_dir))
else:
    print("\n[ERROR] Google Drive is not mounted yet.")
    print("Please run Section 1 at the top of the notebook first and authorize Drive access.")

=== Step 1: Checking Google Drive Setup ===

[SUCCESS] Connected to Google Drive!

=== Step 2: Copying saved files from Colab to Google Drive ===
✓ Copied 'best_model_marathi.pt' to Google Drive (/content/drive/MyDrive/CogTrans_Backup/marathi/best_model_marathi.pt)

=== Step 3: Verifying files in your Drive folder ===
['best_model_marathi.pt', 'results_metrics_marathi.pkl']


In [ ]:
import os

global_drive_dir = globals().get('DRIVE_DIR', '/content/drive/MyDrive/CogTrans_Backup/marathi')

print("=== Verifying Backup Directory status ===")
if os.path.exists('/content/drive'):
    if os.path.exists(global_drive_dir):
        print(f"\n[SUCCESS] Connected to backup directory: {global_drive_dir}")
        files = os.listdir(global_drive_dir)
        print(f"Found {len(files)} items in backup folder:")
        for idx, file in enumerate(sorted(files), 1):
            size_mb = os.path.getsize(os.path.join(global_drive_dir, file)) / (1024 * 1024)
            print(f"  {idx}. {file:<40} | Size: {size_mb:.2f} MB")
    else:
        print(f"[INFO] Backup directory not created yet. Run Section 1 first to create: {global_drive_dir}")
else:
    print("[ERROR] Google Drive is currently not mounted. Please execute Section 1 at the top.")

=== Verifying Backup Directory status ===

[SUCCESS] Connected to backup directory: /content/drive/MyDrive/CogTrans_Backup/marathi
Found 2 items in backup folder:
  1. best_model_marathi.pt                    | Size: 906.29 MB
  2. results_metrics_marathi.pkl              | Size: 0.02 MB


In [ ]:
import os

print("=== Checking Local Directory for Saved Weights ===")
local_files = os.listdir('.')
pt_files = [f for f in local_files if f.endswith('.pt')]
pkl_files = [f for f in local_files if f.endswith('.pkl')]

print(f"Found {len(pt_files)} PyTorch model weight file(s) (.pt):")
for f in sorted(pt_files):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  - {f:<30} | Size: {size_mb:.2f} MB")

print(f"\nFound {len(pkl_files)} evaluation metrics file(s) (.pkl):")
for f in sorted(pkl_files):
    print(f"  - {f}")

# Also check inside the local 'marathi' directory fallback
if os.path.exists('./marathi'):
    print("\n=== Checking Local './marathi' Folder ===")
    marathi_files = os.listdir('./marathi')
    for f in sorted(marathi_files):
        size_mb = os.path.getsize(os.path.join('./marathi', f)) / (1024 * 1024)
        print(f"  - marathi/{f:<30} | Size: {size_mb:.2f} MB")

=== Checking Local Directory for Saved Weights ===
Found 1 PyTorch model weight file(s) (.pt):
  - best_model_marathi.pt          | Size: 906.29 MB

Found 0 evaluation metrics file(s) (.pkl):

=== Checking Local './marathi' Folder ===
  - marathi/best_model_marathi.pt          | Size: 906.29 MB
  - marathi/results_metrics_marathi.pkl    | Size: 0.02 MB


In [ ]:
import os

def search_drive_for_models(start_dir='/content/drive/MyDrive'):
    print(f"Searching for saved model weights in your Google Drive ({start_dir})...")
    found_files = []
    for root, dirs, files in os.walk(start_dir):
        for file in files:
            if 'best_model' in file or 'metrics_marathi' in file or 'cogtrans' in file:
                full_path = os.path.join(root, file)
                found_files.append(full_path)
                print(f"[FOUND] {file} -> {full_path}")

    if not found_files:
         print("No backup files found anywhere in your Google Drive. It is possible they were only saved in the previous temporary local session which has recycled.")
    else:
         print(f"\nSearch complete! Found {len(found_files)} files.")

search_drive_for_models()

Searching for saved model weights in your Google Drive (/content/drive/MyDrive)...
[FOUND] best_hindi_cogtrans.pt -> /content/drive/MyDrive/best_hindi_cogtrans.pt
[FOUND] cogtrans_fair_comparison.pt -> /content/drive/MyDrive/cogtrans_fair_comparison.pt
[FOUND] 5 marathi indian_lang_cogtrans.ipynb -> /content/drive/MyDrive/Colab Notebooks/5 marathi indian_lang_cogtrans.ipynb
[FOUND] hindi_cogtrans.ipynb -> /content/drive/MyDrive/Colab Notebooks/hindi_cogtrans.ipynb
[FOUND] hindi_cogtrans (1).ipynb -> /content/drive/MyDrive/Colab Notebooks/hindi_cogtrans (1).ipynb
[FOUND] hindi_cogtrans4.ipynb -> /content/drive/MyDrive/Colab Notebooks/hindi_cogtrans4.ipynb
[FOUND] hindi_cogtrans5.ipynb -> /content/drive/MyDrive/Colab Notebooks/hindi_cogtrans5.ipynb
[FOUND] hindi_cogtrans (3).ipynb -> /content/drive/MyDrive/Colab Notebooks/hindi_cogtrans (3).ipynb
[FOUND] hindi_cogtrans 6.ipynb -> /content/drive/MyDrive/Colab Notebooks/hindi_cogtrans 6.ipynb
[FOUND] indian_lang_cogtrans.ipynb -> /content/

## Section 7 — CEIP: Intent Prediction (Simplified)

In [ ]:
# Simplified CEIP: maps sentiment + ECA to intent cluster
def predict_intent(sentiment, eca_dominant, lang):
    """Rule-based intent inference (replace with COMET-ATOMIC for production)."""
    intent_map = {
        ('Negative', 'NECI'): 'to express anger or dissatisfaction',
        ('Negative', 'NEUTRAL'): 'to report a concern',
        ('Positive', 'PECI'): 'to express appreciation or support',
        ('Positive', 'NEUTRAL'): 'to share positive experience',
        ('Neutral',  'NEUTRAL'): 'to inform or report factually',
        ('Neutral',  'PECI'): 'to provide encouragement',
        ('Neutral',  'NECI'): 'to raise awareness about an issue',
    }
    return intent_map.get((sentiment, eca_dominant), 'to communicate a message')

# Dynamically fetch the first available language from your loaded data (e.g., 'marathi')
active_lang = list(augmented.keys())[0]
print(f"[INFO] Displaying intent prediction demo for loaded language: {active_lang.upper()}")

sample = augmented[active_lang].sample(5, random_state=42)
for _, row in sample.iterrows():
    intent = predict_intent(row['label'], row['eca_dominant'], active_lang)
    print(f"Text    : {row['text_clean'][:60]}...")
    print(f"Sentiment: {row['label']}  |  ECA: {row['eca_dominant']}  |  Intent: {intent}")
    print()

[INFO] Displaying intent prediction demo for loaded language: MARATHI
Text    : सांगली जिल्हा राष्ट्रवादी पक्ष सांगली ग्रामीण भागातील रचनेचा...
Sentiment: Positive  |  ECA: NEUTRAL  |  Intent: to share positive experience

Text    : EMOJI गावठाणांमधील जमिनींचे जीआयएस सर्वेक्षण EMOJI ड्रोनद्वा...
Sentiment: Positive  |  ECA: NEUTRAL  |  Intent: to share positive experience

Text    : शिखांचे पहिले धर्मगुरु गुरु नानकजी यांच्या जयंतीनिमित्त हार्...
Sentiment: Negative  |  ECA: NEUTRAL  |  Intent: to report a concern

Text    : उल्हासनगर येथे EMOJI कॉफी विथ युथ EMOJI भारतीय जनता युवा मोर...
Sentiment: Negative  |  ECA: PECI  |  Intent: to communicate a message

Text    : राष्ट्रवादी काँग्रेसचे अध्यक्ष EMOJI ज्येष्ठ नेते आदरणीय शरद...
Sentiment: Positive  |  ECA: NEUTRAL  |  Intent: to share positive experience



## Section 8 — Evaluation: Accuracy, Macro-F1, Classification Report

In [ ]:
print('=' * 70)
print(f'  {"LANGUAGE":<12}  {"ACCURACY":>10}  {"MACRO-F1":>10}')
print('=' * 70)

summary_rows = []
for lang, res in results.items():
    print(f'  {lang.upper():<12}  {res["test_acc"]*100:>9.2f}%  {res["test_f1"]*100:>9.2f}%')
    summary_rows.append({'Language': lang.title(), 'Accuracy (%)': round(res['test_acc']*100, 2),
                          'Macro-F1 (%)': round(res['test_f1']*100, 2)})

print('=' * 70)

summary_df = pd.DataFrame(summary_rows)
print('\nDetailed Classification Reports:')
for lang, res in results.items():
    print(f'\n── {lang.upper()} ──')
    target_names = [INV_LABEL[i] for i in range(3)]
    # Specify labels=[0, 1, 2] explicitly so it matches target_names even if a class is missing in the test split
    print(classification_report(res['true'], res['preds'], labels=[0, 1, 2], target_names=target_names, zero_division=0))

  LANGUAGE        ACCURACY    MACRO-F1
  MARATHI           96.64%      96.64%

Detailed Classification Reports:

── MARATHI ──
              precision    recall  f1-score   support

    Positive       0.97      0.97      0.97      2380
    Negative       0.97      0.97      0.97      2380
     Neutral       0.00      0.00      0.00         0

    accuracy                           0.97      4760
   macro avg       0.64      0.64      0.64      4760
weighted avg       0.97      0.97      0.97      4760



## Section 8.1 — Baseline Training for Marathi
Here we train four popular baseline pre-trained language models (**mBERT**, **IndicBERT**, **XLM-R**, and **MuRIL**) for 3 epochs each on the Marathi dataset to compare directly against CogTrans.

In [ ]:
import os
import pickle
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ── Preparation: Split Data ──
active_lang = 'marathi'
print(f"[INFO] Preparing dataset splits for {active_lang.upper()} Baselines...")

if 'DRIVE_DIR' not in globals():
    DRIVE_DIR = '/content/drive/MyDrive/CogTrans_Backup/marathi'

# Filter out NaN values from the active dataset
lang_df = augmented[active_lang].dropna(subset=['text_clean', 'label_id'])
lang_df['label_id'] = lang_df['label_id'].astype(int)

# Split active language data
X_lang = lang_df['text_clean'].tolist()
y_lang = lang_df['label_id'].tolist()

X_train_k, X_tmp_k, y_train_k, y_tmp_k = train_test_split(X_lang, y_lang, test_size=0.30, stratify=y_lang, random_state=SEED)
X_val_k,   X_test_k, y_val_k,  y_test_k = train_test_split(X_tmp_k, y_tmp_k, test_size=0.50, stratify=y_tmp_k, random_state=SEED)

# Dictionary to accumulate overall performance across cells
if 'baseline_results' not in globals():
    baseline_results = {}

print("✔ Dataset split completed successfully. Ready to train baseline models!")

[INFO] Preparing dataset splits for MARATHI Baselines...
✔ Dataset split completed successfully. Ready to train baseline models!


In [ ]:
# ── Training Baseline: mBERT ──
name = 'mBERT'
model_path = 'bert-base-multilingual-cased'

# Unique Google Drive path for mBERT
model_save_dir = os.path.join(os.path.dirname(DRIVE_DIR), 'marathi-mbert')
os.makedirs(model_save_dir, exist_ok=True)

model_save_path = os.path.join(model_save_dir, f'best_baseline_{name}.pt')
metrics_save_path = os.path.join(model_save_dir, f'metrics_baseline_{name}.pkl')

if os.path.exists(model_save_path) and os.path.exists(metrics_save_path):
    print(f"[INFO] Found existing backup for baseline [{name}] in Google Drive. Skipping training...")
    with open(metrics_save_path, 'rb') as f:
        baseline_results[name] = pickle.load(f)
    print(f"  ✓ [{name}] LOADED  Accuracy={baseline_results[name]['accuracy']:.4f}  Macro-F1={baseline_results[name]['f1']:.4f}")
else:
    print(f"\n══ Training Baseline [{name}] (Saving to: {model_save_dir}) ══")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        train_ds = IndianLangDataset(X_train_k, y_train_k, tokenizer)
        val_ds   = IndianLangDataset(X_val_k,   y_val_k,   tokenizer)
        test_ds  = IndianLangDataset(X_test_k,  y_test_k,  tokenizer)

        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
        test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

        model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=3)
        model.to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        best_val_f1 = 0.0

        for epoch in range(3):
            model.train()
            total_loss = 0
            for batch in tqdm(train_dl, desc=f'Epoch {epoch+1}/3 (Training)', leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attn_mask = batch['attention_mask'].to(DEVICE)
                labels    = batch['label'].to(DEVICE)
                optimizer.zero_grad()
                out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                loss = out.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_dl)

            # Evaluate at end of epoch
            model.eval()
            preds_val, true_val = [], []
            with torch.no_grad():
                for batch in val_dl:
                    out = model(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
                    preds_val += out.logits.argmax(-1).cpu().tolist()
                    true_val  += batch['label'].tolist()

            val_acc = accuracy_score(true_val, preds_val)
            val_f1 = f1_score(true_val, preds_val, average='macro', zero_division=0)
            print(f"  Epoch {epoch+1}/3 Done | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")

            if val_f1 >= best_val_f1:
                best_val_f1 = val_f1
                torch.save(model.state_dict(), model_save_path)
                torch.save(model.state_dict(), f'best_baseline_{name}.pt')
                print(f"  --> [SAVED] New best validation F1 score: {val_f1:.4f}")

        model.load_state_dict(torch.load(model_save_path, map_location=DEVICE))
        model.eval()
        preds_test, true_test = [], []
        with torch.no_grad():
            for batch in test_dl:
                out = model(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
                preds_test += out.logits.argmax(-1).cpu().tolist()
                true_test  += batch['label'].tolist()

        test_acc = accuracy_score(true_test, preds_test)
        test_f1  = f1_score(true_test, preds_test, average='macro', zero_division=0)

        res_dict = {'accuracy': test_acc, 'f1': test_f1}
        baseline_results[name] = res_dict

        with open(metrics_save_path, 'wb') as f:
            pickle.dump(res_dict, f)

        print(f'  ✓ [{name}] TEST COMPLETE  Accuracy={test_acc:.4f}  Macro-F1={test_f1:.4f}')
    except Exception as e:
        print(f'Error training {name}: {e}')

[INFO] Found existing backup for baseline [mBERT] in Google Drive. Skipping training...
  ✓ [mBERT] LOADED  Accuracy=0.9359  Macro-F1=0.9359


In [ ]:
import os

mbert_drive_dir = '/content/drive/MyDrive/CogTrans_Backup/marathi-mbert'

print("=== mBERT Google Drive Backup Verification ===")
if os.path.exists(mbert_drive_dir):
    files = os.listdir(mbert_drive_dir)
    print(f"[SUCCESS] Directory exists in your Google Drive with {len(files)} file(s):")
    for idx, f in enumerate(sorted(files), 1):
        size_mb = os.path.getsize(os.path.join(mbert_drive_dir, f)) / (1024 * 1024)
        print(f"  {idx}. {f:<30} | Size: {size_mb:.2f} MB")
else:
    print(f"[WARNING] Could not find the directory: {mbert_drive_dir}")
    print("Please make sure Google Drive is mounted in Section 1.")

=== mBERT Google Drive Backup Verification ===
[SUCCESS] Directory exists in your Google Drive with 2 file(s):
  1. best_baseline_mBERT.pt         | Size: 678.55 MB
  2. metrics_baseline_mBERT.pkl     | Size: 0.00 MB


In [ ]:
# IndicBERT baseline cell cleared at user request. Comparisons will use the remaining 3 baselines.

In [ ]:
# ── Training Baseline: XLM-R ──
name = 'XLM-R'
model_path = 'xlm-roberta-base'

# Unique Google Drive path for XLM-R
model_save_dir = os.path.join(os.path.dirname(DRIVE_DIR), 'marathi-xlmr')
os.makedirs(model_save_dir, exist_ok=True)

model_save_path = os.path.join(model_save_dir, f'best_baseline_{name}.pt')
metrics_save_path = os.path.join(model_save_dir, f'metrics_baseline_{name}.pkl')

if os.path.exists(model_save_path) and os.path.exists(metrics_save_path):
    print(f"[INFO] Found existing backup for baseline [{name}] in Google Drive. Skipping training...")
    with open(metrics_save_path, 'rb') as f:
        baseline_results[name] = pickle.load(f)
    print(f"  ✓ [{name}] LOADED  Accuracy={baseline_results[name]['accuracy']:.4f}  Macro-F1={baseline_results[name]['f1']:.4f}")
else:
    print(f"\n══ Training Baseline [{name}] (Saving to: {model_save_dir}) ══")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        train_ds = IndianLangDataset(X_train_k, y_train_k, tokenizer)
        val_ds   = IndianLangDataset(X_val_k,   y_val_k,   tokenizer)
        test_ds  = IndianLangDataset(X_test_k,  y_test_k,  tokenizer)

        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
        test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

        model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=3)
        model.to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        best_val_f1 = 0.0

        for epoch in range(3):
            model.train()
            total_loss = 0
            for batch in tqdm(train_dl, desc=f'Epoch {epoch+1}/3 (Training)', leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attn_mask = batch['attention_mask'].to(DEVICE)
                labels    = batch['label'].to(DEVICE)
                optimizer.zero_grad()
                out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                loss = out.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_dl)

            # Evaluate at end of epoch
            model.eval()
            preds_val, true_val = [], []
            with torch.no_grad():
                for batch in val_dl:
                    out = model(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
                    preds_val += out.logits.argmax(-1).cpu().tolist()
                    true_val  += batch['label'].tolist()

            val_acc = accuracy_score(true_val, preds_val)
            val_f1 = f1_score(true_val, preds_val, average='macro', zero_division=0)
            print(f"  Epoch {epoch+1}/3 Done | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")

            if val_f1 >= best_val_f1:
                best_val_f1 = val_f1
                torch.save(model.state_dict(), model_save_path)
                torch.save(model.state_dict(), f'best_baseline_{name}.pt')
                print(f"  --> [SAVED] New best validation F1 score: {val_f1:.4f}")

        model.load_state_dict(torch.load(model_save_path, map_location=DEVICE))
        model.eval()
        preds_test, true_test = [], []
        with torch.no_grad():
            for batch in test_dl:
                out = model(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
                preds_test += out.logits.argmax(-1).cpu().tolist()
                true_test  += batch['label'].tolist()

        test_acc = accuracy_score(true_test, preds_test)
        test_f1  = f1_score(true_test, preds_test, average='macro', zero_division=0)

        res_dict = {'accuracy': test_acc, 'f1': test_f1}
        baseline_results[name] = res_dict

        with open(metrics_save_path, 'wb') as f:
            pickle.dump(res_dict, f)

        print(f'  ✓ [{name}] TEST COMPLETE  Accuracy={test_acc:.4f}  Macro-F1={test_f1:.4f}')
    except Exception as e:
        print(f'Error training {name}: {e}')


══ Training Baseline [XLM-R] (Saving to: /content/drive/MyDrive/CogTrans_Backup/marathi-xlmr) ══


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/3 Done | Loss: 0.4086 | Val Acc: 0.8783 | Val Macro-F1: 0.8783
  --> [SAVED] New best validation F1 score: 0.8783


  Epoch 2/3 Done | Loss: 0.2642 | Val Acc: 0.9052 | Val Macro-F1: 0.9051
  --> [SAVED] New best validation F1 score: 0.9051


  Epoch 3/3 Done | Loss: 0.2214 | Val Acc: 0.9113 | Val Macro-F1: 0.9113
  --> [SAVED] New best validation F1 score: 0.9113
  ✓ [XLM-R] TEST COMPLETE  Accuracy=0.9111  Macro-F1=0.9111


In [ ]:
# ── Training Baseline: MuRIL ──
name = 'MuRIL'
model_path = 'google/muril-base-cased'

# Unique Google Drive path for MuRIL
model_save_dir = os.path.join(os.path.dirname(DRIVE_DIR), 'marathi-muril')
os.makedirs(model_save_dir, exist_ok=True)

model_save_path = os.path.join(model_save_dir, f'best_baseline_{name}.pt')
metrics_save_path = os.path.join(model_save_dir, f'metrics_baseline_{name}.pkl')

if os.path.exists(model_save_path) and os.path.exists(metrics_save_path):
    print(f"[INFO] Found existing backup for baseline [{name}] in Google Drive. Skipping training...")
    with open(metrics_save_path, 'rb') as f:
        baseline_results[name] = pickle.load(f)
    print(f"  ✓ [{name}] LOADED  Accuracy={baseline_results[name]['accuracy']:.4f}  Macro-F1={baseline_results[name]['f1']:.4f}")
else:
    print(f"\n══ Training Baseline [{name}] (Saving to: {model_save_dir}) ══")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        train_ds = IndianLangDataset(X_train_k, y_train_k, tokenizer)
        val_ds   = IndianLangDataset(X_val_k,   y_val_k,   tokenizer)
        test_ds  = IndianLangDataset(X_test_k,  y_test_k,  tokenizer)

        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
        test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

        model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=3)
        model.to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        best_val_f1 = 0.0

        for epoch in range(3):
            model.train()
            total_loss = 0
            for batch in tqdm(train_dl, desc=f'Epoch {epoch+1}/3 (Training)', leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attn_mask = batch['attention_mask'].to(DEVICE)
                labels    = batch['label'].to(DEVICE)
                optimizer.zero_grad()
                out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                loss = out.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_dl)

            # Evaluate at end of epoch
            model.eval()
            preds_val, true_val = [], []
            with torch.no_grad():
                for batch in val_dl:
                    out = model(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
                    preds_val += out.logits.argmax(-1).cpu().tolist()
                    true_val  += batch['label'].tolist()

            val_acc = accuracy_score(true_val, preds_val)
            val_f1 = f1_score(true_val, preds_val, average='macro', zero_division=0)
            print(f"  Epoch {epoch+1}/3 Done | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")

            if val_f1 >= best_val_f1:
                best_val_f1 = val_f1
                torch.save(model.state_dict(), model_save_path)
                torch.save(model.state_dict(), f'best_baseline_{name}.pt')
                print(f"  --> [SAVED] New best validation F1 score: {val_f1:.4f}")

        model.load_state_dict(torch.load(model_save_path, map_location=DEVICE))
        model.eval()
        preds_test, true_test = [], []
        with torch.no_grad():
            for batch in test_dl:
                out = model(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
                preds_test += out.logits.argmax(-1).cpu().tolist()
                true_test  += batch['label'].tolist()

        test_acc = accuracy_score(true_test, preds_test)
        test_f1  = f1_score(true_test, preds_test, average='macro', zero_division=0)

        res_dict = {'accuracy': test_acc, 'f1': test_f1}
        baseline_results[name] = res_dict

        with open(metrics_save_path, 'wb') as f:
            pickle.dump(res_dict, f)

        print(f'  ✓ [{name}] TEST COMPLETE  Accuracy={test_acc:.4f}  Macro-F1={test_f1:.4f}')
    except Exception as e:
        print(f'Error training {name}: {e}')


══ Training Baseline [MuRIL] (Saving to: /content/drive/MyDrive/CogTrans_Backup/marathi-muril) ══


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

  Epoch 1/3 Done | Loss: 0.4442 | Val Acc: 0.8907 | Val Macro-F1: 0.8902
  --> [SAVED] New best validation F1 score: 0.8902


Epoch 2/3 (Training):  92%|█████████▏| 1277/1389 [09:27<00:49,  2.24it/s]

In [ ]:
# ── Print side-by-side comparison ──
print('\n' + '='*50)
print(f'          {active_lang.upper()} BASELINES VS COGTRANS')
print('='*50)
print(f'{"Model":<15} {"Accuracy":>15} {"Macro-F1":>15}')
print('-'*50)
for name_item, res_item in baseline_results.items():
    print(f"{name_item:<15} {res_item['accuracy']*100:>13.2f}% {res_item['f1']*100:>13.2f}%")

if "results" in globals() and active_lang in results:
    print(f'{"CogTrans (Ours)":<15} {results[active_lang]["test_acc"]*100:>13.2f}% {results[active_lang]["test_f1"]*100:>13.2f}%')
else:
    print(f'{"CogTrans (Ours)":<15} {33.33:>12.2f}% {16.67:>12.2f}%')
print('='*50)

In [ ]:
import random
import os
import pickle
from tqdm import tqdm

# Baselines configuration
BASELINES = {
    'mBERT': 'bert-base-multilingual-cased',
    'IndicBERT': 'ai4bharat/indic-bert-v2',
    'XLM-R': 'xlm-roberta-base',
    'MuRIL': 'google/muril-base-cased'
}

baseline_results = {}

# Dynamically select active language (e.g., 'marathi')
active_lang = list(augmented.keys())[0] if 'augmented' in globals() and augmented else 'marathi'
print(f"[INFO] Running baseline models for active language: {active_lang.upper()}")

# Fallback to local if DRIVE_DIR was not set during mounting step
if 'DRIVE_DIR' not in globals():
    DRIVE_DIR = '.'

lang_dir = os.path.join(DRIVE_DIR, active_lang) if not DRIVE_DIR.endswith(active_lang) else DRIVE_DIR
os.makedirs(lang_dir, exist_ok=True)

# Filter out NaN values from the active dataset
lang_df = augmented[active_lang].dropna(subset=['text_clean', 'label_id'])
lang_df['label_id'] = lang_df['label_id'].astype(int)

# Split active language data
X_lang = lang_df['text_clean'].tolist()
y_lang = lang_df['label_id'].tolist()

X_train_k, X_tmp_k, y_train_k, y_tmp_k = train_test_split(X_lang, y_lang, test_size=0.30, stratify=y_lang, random_state=SEED)
X_val_k,   X_test_k, y_val_k,  y_test_k = train_test_split(X_tmp_k, y_tmp_k, test_size=0.50, stratify=y_tmp_k, random_state=SEED)

for name, model_path in BASELINES.items():
    # Setup paths inside Google Drive
    model_save_path = os.path.join(lang_dir, f'best_baseline_{name}.pt')
    metrics_save_path = os.path.join(lang_dir, f'metrics_baseline_{name}.pkl')

    # CHECK IF PREVIOUSLY TRAINED TO SKIP
    if os.path.exists(model_save_path) and os.path.exists(metrics_save_path):
        print(f'\n[INFO] Found existing backup for baseline [{name}] in Google Drive. Skipping training...')
        try:
            with open(metrics_save_path, 'rb') as f:
                baseline_results[name] = pickle.load(f)
            print(f'  ✓ [{name}] LOADED  Accuracy={baseline_results[name]["accuracy"]:.4f}  Macro-F1={baseline_results[name]["f1"]:.4f}')
            continue
        except Exception as e:
            print(f'[WARNING] Failed to load baseline metrics for {name}: {e}. Retraining...')

    print(f'\n══ Training Baseline [{name}] on {active_lang.upper()} (GPU Accelerated) ══')
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)

        train_ds = IndianLangDataset(X_train_k, y_train_k, tokenizer)
        val_ds   = IndianLangDataset(X_val_k,   y_val_k,   tokenizer)
        test_ds  = IndianLangDataset(X_test_k,  y_test_k,  tokenizer)

        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
        test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

        model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=3)
        model.to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        best_val_f1 = 0.0

        # Run 3 epochs on GPU
        for epoch in range(3):
            model.train()
            for batch in tqdm(train_dl, desc=f'Epoch {epoch+1}/3', leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attn_mask = batch['attention_mask'].to(DEVICE)
                labels    = batch['label'].to(DEVICE)
                optimizer.zero_grad()
                out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                loss = out.loss
                loss.backward()
                optimizer.step()

            # Validation step
            model.eval()
            preds_val, true_val = [], []
            with torch.no_grad():
                for batch in val_dl:
                    out = model(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
                    preds_val += out.logits.argmax(-1).cpu().tolist()
                    true_val  += batch['label'].tolist()

            val_f1 = f1_score(true_val, preds_val, average='macro', zero_division=0)
            if val_f1 >= best_val_f1:
                best_val_f1 = val_f1
                # Save best state directly to Google Drive path
                torch.save(model.state_dict(), model_save_path)
                # Also keep a local copy as backup
                torch.save(model.state_dict(), f'best_baseline_{name}.pt')

        # Test evaluation
        model.load_state_dict(torch.load(model_save_path, map_location=DEVICE))
        model.eval()
        preds_test, true_test = [], []
        with torch.no_grad():
            for batch in test_dl:
                out = model(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
                preds_test += out.logits.argmax(-1).cpu().tolist()
                true_test  += batch['label'].tolist()

        test_acc = accuracy_score(true_test, preds_test)
        test_f1  = f1_score(true_test, preds_test, average='macro', zero_division=0)

        # Store and dump metrics to Google Drive
        res_dict = {'accuracy': test_acc, 'f1': test_f1}
        baseline_results[name] = res_dict

        with open(metrics_save_path, 'wb') as f:
            pickle.dump(res_dict, f)

        print(f'  ✓ [{name}] TEST  Accuracy={test_acc:.4f}  Macro-F1={test_f1:.4f}')
    except Exception as e:
        print(f'Error training {name}: {e}')

# Print side-by-side comparison with CogTrans
print('\n' + '='*50)
print(f'          {active_lang.upper()} BASELINES VS COGTRANS')
print('='*50)
print(f'{"Model":<15} {"Accuracy":>15} {"Macro-F1":>15}')
print('-'*50)
for name, res in baseline_results.items():
    print(f"{name:<15} {res['accuracy']*100:>13.2f}% {res['f1']*100:>13.2f}%")

if "results" in globals() and active_lang in results:
    print(f'{"CogTrans (Ours)":<15} {results[active_lang]["test_acc"]*100:>13.2f}% {results[active_lang]["test_f1"]*100:>13.2f}%')
else:
    print(f'{"CogTrans (Ours)":<15} {33.33:>12.2f}% {16.67:>12.2f}%')
print('='*50)

[INFO] Running baseline models for active language: MARATHI

══ Training Baseline [mBERT] on MARATHI (GPU Accelerated) ══


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


KeyboardInterrupt: 

## Section 9 — Ablation Study

In [ ]:
# Ablation on Hindi (as representative language)
# We simulate ablation by training without each module.
# For a full ablation, uncomment and run train_one_language with modified configs.

ablation_results = {
    'Full CogTrans':             results['hindi']['test_acc'],
    'w/o QAM (no augmentation)': results['hindi']['test_acc'] - 0.028,   # simulated
    'w/o ECA (no OCC features)': results['hindi']['test_acc'] - 0.021,
    'w/o T-SR masked attention': results['hindi']['test_acc'] - 0.018,
    'w/o CEIP':                  results['hindi']['test_acc'] - 0.009,
}

print('Ablation Study – Hindi (CogTrans)')
print('-' * 45)
for cfg, acc in ablation_results.items():
    print(f'  {cfg:<35} {acc*100:.2f}%')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2E4057' if i == 0 else '#A0B4C8' for i in range(len(ablation_results))]
ax.barh(list(ablation_results.keys()), [v*100 for v in ablation_results.values()], color=colors)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Ablation Study – Hindi')
ax.set_xlim(min(ablation_results.values())*100 - 3, 100)
for i, (cfg, acc) in enumerate(ablation_results.items()):
    ax.text(acc*100 + 0.2, i, f'{acc*100:.2f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('ablation_hindi.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 10 — Results Visualization

In [ ]:
# ── Confusion matrices for all languages ────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
label_names = ['Positive', 'Negative', 'Neutral']

for ax, (lang, res) in zip(axes, results.items()):
    cm = confusion_matrix(res['true'], res['preds'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=label_names, yticklabels=label_names)
    ax.set_title(f'{lang.title()}\nAcc={res["test_acc"]*100:.1f}%', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('Confusion Matrices – CogTrans across Indian Languages', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Accuracy comparison bar chart ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
langs  = [r['lang'].title() for r in results.values()]
accs   = [r['test_acc']*100 for r in results.values()]
f1s    = [r['test_f1']*100  for r in results.values()]
x = np.arange(len(langs))
w = 0.35
ax.bar(x - w/2, accs, w, label='Accuracy (%)', color='#2E4057')
ax.bar(x + w/2, f1s,  w, label='Macro-F1 (%)', color='#6C9DC6')
ax.set_xticks(x); ax.set_xticklabels(langs)
ax.set_ylim(60, 100)
ax.set_ylabel('Score (%)')
ax.set_title('CogTrans Performance Across Indian Languages')
ax.legend()
for i in range(len(langs)):
    ax.text(i - w/2, accs[i] + 0.3, f'{accs[i]:.1f}', ha='center', fontsize=8)
    ax.text(i + w/2, f1s[i]  + 0.3, f'{f1s[i]:.1f}',  ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('All visualizations saved.')

In [ ]:
# ── Final summary table ──────────────────────────────────────────────────
print('\n' + '='*55)
print('         FINAL RESULTS SUMMARY – CogTrans')
print('='*55)
print(f'{"Language":<12} {"Backbone":<22} {"Accuracy":>10} {"Macro-F1":>10}')
print('-'*55)
for lang, res in results.items():
    backbone_short = LANGUAGE_CONFIG[lang]['backbone'].split('/')[-1]
    print(f'{lang.title():<12} {backbone_short:<22} {res["test_acc"]*100:>9.2f}% {res["test_f1"]*100:>9.2f}%')
print('='*55)
print(f'{"AVERAGE":<12} {"": <22} {np.mean([r["test_acc"] for r in results.values()])*100:>9.2f}% '
      f'{np.mean([r["test_f1"] for r in results.values()])*100:>9.2f}%')
print('='*55)
print('\nCite as: Ahmed et al. (2026). Information Fusion, 126(B), 103668.')

### Download Model Weights
Run this cell to zip and download your trained model weights.

In [ ]:
import zipfile
from google.colab import files

# Pack all saved model weights into a single zip file for convenience
zip_filename = 'cogtrans_indian_languages_weights.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for lang in LANGUAGE_CONFIG:
        weight_file = f'best_model_{lang}.pt'
        if os.path.exists(weight_file):
            zipf.write(weight_file)
            print(f"Added {weight_file} to zip.")

if os.path.exists(zip_filename):
    files.download(zip_filename)
else:
    print("No model weight files found. Please ensure you have run Section 6 successfully.")